# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [2]:
# !python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [3]:
# # https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
# ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# # https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
# ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# # https://platform.openai.com/api-keys
# OPENAI_API_KEY = getpass("OpenAI API key: ")
import os 
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
ELASTIC_API_KEY = os.environ.get("ELASTIC_API_KEY")
ELASTIC_CLOUD_ID = os.environ.get("ELASTIC_CLOUD_ID")

embeddings = OpenAIEmbeddings()

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="my-index",
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [4]:
# from urllib.request import urlopen
# import json

# url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

# response = urlopen(url)
# data = json.load(response)

# with open("temp.json", "w") as json_file:
#     json.dump(data, json_file)

In [5]:
import json
import requests

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    
    data = response.json()

    with open("temp.json", "w") as json_file:
        json.dump(data, json_file)
        
    print("✅ Success! File downloaded and saved to temp.json.")

except requests.exceptions.Timeout:
    print("❌ Error: The connection timed out. Check your internet connection or VPN.")
except requests.exceptions.RequestException as e:
    print(f"❌ Error during download: {e}")

✅ Success! File downloaded and saved to temp.json.


### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [6]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=50 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

/var/folders/gn/wg9xjv455tj2c2jns2j4b8yw0000gn/T/ipykernel_63786/645203412.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [7]:
from langchain_openai import ChatOpenAI
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="my-index",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = ChatOpenAI(temperature=0)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [8]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.prompts import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

---- Answer ----
content="The NASA sales team refers to the North America South America region within the sales organization of the company. This region includes the United States, Canada, Mexico, as well as Central and South America. The NASA sales team is divided into two areas, each led by an Area Vice-President. Laura Martinez is the Area Vice-President of North America, and Gary Johnson is the Area Vice-President of South America. \n\nThe NASA sales team is responsible for serving customers in the North America and South America regions, identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction. The team collaborates closely with other departments within the organization, such as marketing, product development, and customer support, to deliver high-quality products and services to clients in these regions.\n\nOverall, the NASA sales team plays a crucial role in achieving the company's business objectives across

**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [9]:
import logging
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, format_document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are a strict, factual technical analyst. Analyze the provided context and answer the question using clear, concise markdown bullet points. 
    If the context does not contain the answer, explicitly state "Information not found in database." and say nothing else. Do not extrapolate or guess.
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain_factual = _context | LLM_CONTEXT_PROMPT | llm

ans = chain_factual.invoke("what are the specific criteria for joining the nasa sales team?")

print("---- Executive Fact-Check ----")
print(ans)

---- Executive Fact-Check ----
content='- The NASA sales team is divided into two regions: North America and South America\n- Laura Martinez is the Area Vice-President of North America\n- Gary Johnson is the Area Vice-President of South America' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 710, 'total_tokens': 750, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DlDlXhLBvkaMqS3I8wf8j6S5dVgrv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e7903-0be4-7542-9b7b-efde0b024046-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 710, 'output_tokens': 40, 'total_tokens': 750, 'input_token_details': {'audio':

In [10]:
import logging
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, format_document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful, friendly, and patient customer support representative for our company. 
    Review the internal documentation context below and answer the customer's question in a warm, welcoming tone.
    Break down complex terms into simple language. If you don't know the answer, politely let them know you'll look into it.
    
    context: {context}
    Customer Question: "{question}"
    Support Agent Response:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
DOCUMENT: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain_support = _context | LLM_CONTEXT_PROMPT | llm

ans = chain_support.invoke("hi, i was wondering if you could explain what the nasa sales team actually does?")

print("---- Support Response ----")
print(ans)

---- Support Response ----
content="Hello! Thank you for reaching out with your question about the NASA sales team. The NASA sales team is actually a combination of the North America and South America regions within our sales organization. \n\nIn simple terms, the NASA sales team, led by Area Vice-Presidents Laura Martinez and Gary Johnson, is responsible for identifying new business opportunities, nurturing relationships with existing clients, and ensuring customer satisfaction in both North and South America. They work closely with account managers, sales representatives, and support staff to deliver high-quality products and services to our clients in these regions.\n\nIf you have any more questions or need further clarification, feel free to ask. I'm here to help!" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 134, 'prompt_tokens': 834, 'total_tokens': 968, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0